In [0]:
# ============================================
# CityFlow — 03_gold_layer
# ============================================
import urllib.parse

# Secure credentials
ACCESS_KEY = dbutils.secrets.get(scope="cityflow-secrets", key="aws-access-key")
SECRET_KEY = dbutils.secrets.get(scope="cityflow-secrets", key="aws-secret-key")
encoded_secret = urllib.parse.quote(SECRET_KEY, safe="")

S3_BUCKET = "cityflow-taxi-data"
SILVER_PATH = f"s3a://{ACCESS_KEY}:{encoded_secret}@{S3_BUCKET}/silver"
GOLD_PATH = f"s3a://{ACCESS_KEY}:{encoded_secret}@{S3_BUCKET}/gold"

# Read from Silver
df_silver = spark.read.format("delta") \
    .load(f"{SILVER_PATH}/taxi/")

print(f"Silver records loaded: {df_silver.count()}")
df_silver.printSchema()

Silver records loaded: 10836815
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = 

In [0]:
from pyspark.sql.functions import count, avg, sum, round as spark_round

df_gold_daily = df_silver.groupBy("trip_date") \
    .agg(
        count("*").alias("total_trips"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance"),
        spark_round(sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("tip_amount"), 2).alias("avg_tip")
    ).orderBy("trip_date")

df_gold_daily.show(10)

df_gold_daily.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/daily_summary/")

print("Gold daily summary saved!")

+----------+-----------+--------+------------+-------------+-------+
| trip_date|total_trips|avg_fare|avg_distance|total_revenue|avg_tip|
+----------+-----------+--------+------------+-------------+-------+
|2016-01-01|     342429|    12.8|        3.29|   5331521.17|   1.44|
|2016-01-02|     310835|   12.39|        3.08|   4705944.98|   1.46|
|2016-01-03|     300657|    13.0|        3.43|   4817302.06|   1.65|
|2016-01-04|     314031|   12.16|       30.52|    4809186.6|   1.63|
|2016-01-05|     341071|   11.96|        7.77|   5129983.73|   1.65|
|2016-01-06|     346387|   11.96|         2.8|   5221059.76|   1.67|
|2016-01-07|     362724|   12.09|        2.82|    5538170.9|   1.73|
|2016-01-08|     389711|    12.0|        2.76|   5891605.29|   1.69|
|2016-01-09|     403396|   11.54|        2.71|    5758254.6|   1.54|
|2016-01-10|     349534|   12.14|        3.09|    5279527.7|   1.67|
+----------+-----------+--------+------------+-------------+-------+
only showing top 10 rows
Gold dail

In [0]:
df_gold_hourly = df_silver.groupBy("pickup_hour", "pickup_day") \
    .agg(
        count("*").alias("total_trips"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare")
    ).orderBy("pickup_hour", "pickup_day")

df_gold_hourly.show(10)

df_gold_hourly.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/hourly_demand/")

print("Gold hourly demand saved!")

+-----------+----------+-----------+--------+
|pickup_hour|pickup_day|total_trips|avg_fare|
+-----------+----------+-----------+--------+
|          0|         1|      90260|   11.99|
|          0|         2|      28070|   14.41|
|          0|         3|      30155|   15.03|
|          0|         4|      33164|   13.69|
|          0|         5|      37931|   13.28|
|          0|         6|      77763|   13.02|
|          0|         7|      95504|   12.68|
|          1|         1|      79034|   11.94|
|          1|         2|      17998|   14.16|
|          1|         3|      17066|   14.35|
+-----------+----------+-----------+--------+
only showing top 10 rows
Gold hourly demand saved!
